<a href="https://colab.research.google.com/github/vikiiiiiiiiiiii/Machine-Learning-Projects/blob/main/Transforming_Raw_IoT_Data_into_AI_Powered_Predictive_Maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import time
import json
from datetime import datetime

class IndustrialSensorCluster:
    """Simulates a cluster of factory machine sensors with live telemetry."""

    def __init__(self, cluster_id: str):
        self.cluster_id = cluster_id
        self.base_temp = 65.0       # Celsius
        self.base_vibration = 2.4   # mm/s
        self.base_voltage = 230.0   # Volts

    def generate_telemetry(self) -> dict:
        """Generates a single packet of streaming sensor data, occasionally injecting anomalies."""
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Add random normal operational noise
        temperature = self.base_temp + random.uniform(-2.0, 2.0)
        vibration = self.base_vibration + random.uniform(-0.3, 0.3)
        voltage = self.base_voltage + random.uniform(-5.0, 5.0)
        status = "OPERATIONAL"

        # Strategic Anomaly Injection (10% chance of a system failure event)
        if random.random() < 0.10:
            anomaly_type = random.choice(["OVERHEAT", "VIBRATION_SPIKE", "VOLTAGE_DROP"])
            if anomaly_type == "OVERHEAT":
                temperature += random.uniform(25.0, 40.0)
                status = "CRITICAL_TEMP"
            elif anomaly_type == "VIBRATION_SPIKE":
                vibration += random.uniform(3.5, 5.0)
                status = "CRITICAL_VIB"
            elif anomaly_type == "VOLTAGE_DROP":
                voltage -= random.uniform(40.0, 60.0)
                status = "CRITICAL_VOLT"

        return {
            "timestamp": now,
            "cluster_id": self.cluster_id,
            "telemetry": {
                "temperature_c": round(temperature, 2),
                "vibration_mms": round(vibration, 2),
                "voltage_v": round(voltage, 2)
            },
            "system_status": status
        }

# --- TEST THE STREAM DIRECTLY IN COLAB ---
cluster = IndustrialSensorCluster(cluster_id="cluster_mumbai_01")
print("--- Starting Live Telemetry Stream (Will automatically stop after 10 prints) ---")

# We run it 10 times so it doesn't infinite-loop your Colab cell
for _ in range(10):
    data_packet = cluster.generate_telemetry()
    print(json.dumps(data_packet, indent=2))
    time.sleep(1.0)

--- Starting Live Telemetry Stream (Will automatically stop after 10 prints) ---
{
  "timestamp": "2026-06-13 10:21:55",
  "cluster_id": "cluster_mumbai_01",
  "telemetry": {
    "temperature_c": 65.66,
    "vibration_mms": 2.13,
    "voltage_v": 233.76
  },
  "system_status": "OPERATIONAL"
}
{
  "timestamp": "2026-06-13 10:21:56",
  "cluster_id": "cluster_mumbai_01",
  "telemetry": {
    "temperature_c": 66.45,
    "vibration_mms": 2.2,
    "voltage_v": 228.24
  },
  "system_status": "OPERATIONAL"
}
{
  "timestamp": "2026-06-13 10:21:57",
  "cluster_id": "cluster_mumbai_01",
  "telemetry": {
    "temperature_c": 92.83,
    "vibration_mms": 2.54,
    "voltage_v": 228.86
  },
  "system_status": "CRITICAL_TEMP"
}
{
  "timestamp": "2026-06-13 10:21:58",
  "cluster_id": "cluster_mumbai_01",
  "telemetry": {
    "temperature_c": 65.47,
    "vibration_mms": 2.64,
    "voltage_v": 233.55
  },
  "system_status": "OPERATIONAL"
}
{
  "timestamp": "2026-06-13 10:21:59",
  "cluster_id": "cluster_m

In [3]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

class ProductionDataPipeline:
    """Captures streaming telemetry, flattens it, and structures it for ML training."""

    def __init__(self, storage_path: str = "production_database.csv"):
        self.storage_path = storage_path
        # The clean matrix features our AI will learn from
        self.columns = ["timestamp", "temperature_c", "vibration_mms", "voltage_v", "is_anomaly"]

    def process_packet(self, packet: dict) -> dict:
        """Flattens incoming nested JSON packets and performs binary target labeling."""
        timestamp = packet["timestamp"]
        temp = packet["telemetry"]["temperature_c"]
        vib = packet["telemetry"]["vibration_mms"]
        volt = packet["telemetry"]["voltage_v"]

        # Target Encoding: 1 = System Anomaly/Failure, 0 = Normal Operation
        is_anomaly = 1 if packet["system_status"] != "OPERATIONAL" else 0

        return {
            "timestamp": timestamp,
            "temperature_c": temp,
            "vibration_mms": vib,
            "voltage_v": volt,
            "is_anomaly": is_anomaly
        }

    def save_to_ledger(self, flat_data: dict):
        """Transforms dictionary data into an optimized DataFrame row and appends to storage."""
        df_row = pd.DataFrame([flat_data])

        # Check if database file exists. If not, write with headers. If yes, append raw rows.
        if not os.path.isfile(self.storage_path):
            df_row.to_csv(self.storage_path, index=False)
        else:
            df_row.to_csv(self.storage_path, mode='a', header=False, index=False)

# --- EXECUTE INGESTION PIPELINE ---
pipeline = ProductionDataPipeline()

print("--- Data Pipeline Active: Harvesting Stream Packets ---")
print("Collecting 100 structured telemetry points for AI Model optimization...")

for i in range(100):
    # Call the live generator instance from your Component 1 cell
    raw_packet = cluster.generate_telemetry()
    clean_packet = pipeline.process_packet(raw_packet)
    pipeline.save_to_ledger(clean_packet)

print("\n--- Harvesting Cycle Complete! ---")
print(f"Dataset successfully compiled into: '{pipeline.storage_path}' (100 rows generated)")

--- Data Pipeline Active: Harvesting Stream Packets ---

--- Harvesting Cycle Complete! ---
Dataset successfully compiled into: 'production_database.csv' (100 rows generated)


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import pickle

class PredictiveModelTrainer:
    """Loads compiled production logs, trains a Random Forest classifier, and evaluates metrics."""

    def __init__(self, data_path: str = "production_database.csv"):
        self.data_path = data_path
        # Enforcing an ensemble tree-based algorithm for non-linear anomaly boundaries
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)

    def train_ai_engine(self):
        # 1. Ingest the data matrix you engineered in Component 2
        print("Reading production database matrix...")
        df = pd.read_csv(self.data_path)

        # 2. Feature-Target Matrix Separation
        # X = Input vectors (Measurements), y = Target vector (0 or 1)
        X = df[["temperature_c", "vibration_mms", "voltage_v"]]
        y = df["is_anomaly"]

        # 3. Validation Split (80% Training evaluation space, 20% Unseen verification space)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        print(f"Training AI Engine on {len(X_train)} historical data vectors...")
        self.model.fit(X_train, y_train)

        # 4. Out-of-Sample Verification Testing
        predictions = self.model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        print(f"\n--- AI System Training Complete ---")
        print(f"Model Verification Accuracy: {accuracy * 100:.2f}%")
        print("\nDetailed Performance Metrics Matrix:")
        print(classification_report(y_test, predictions, zero_division=0))

        # 5. Serialization: Exporting mathematical weights to disk
        model_filename = "predictive_model.pkl"
        with open(model_filename, "wb") as file:
            pickle.dump(self.model, file)
        print(f"\nModel artifact successfully serialized and exported as '{model_filename}'")

# --- EXECUTE THE TRAINER ---
trainer = PredictiveModelTrainer()
trainer.train_ai_engine()

Reading production database matrix...
Training AI Engine on 80 historical data vectors...

--- AI System Training Complete ---
Model Verification Accuracy: 95.00%

Detailed Performance Metrics Matrix:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       0.00      0.00      0.00         1

    accuracy                           0.95        20
   macro avg       0.47      0.50      0.49        20
weighted avg       0.90      0.95      0.93        20


Model artifact successfully serialized and exported as 'predictive_model.pkl'


In [1]:
import pickle
import pandas as pd
from pydantic import BaseModel
import json

# 1. Define the input validation schema exactly as we did for FastAPI
class TelemetryInput(BaseModel):
    temperature_c: float
    vibration_mms: float
    voltage_v: float

# 2. Re-verify the memory-mapped model brain is active
MODEL_PATH = "predictive_model.pkl"
try:
    with open(MODEL_PATH, "rb") as file:
        loaded_model = pickle.load(file)
    print(f"✅ Production Success: Model artifact '{MODEL_PATH}' successfully memory-mapped.")
except FileNotFoundError:
    print("❌ Error: Run Component 3 first to generate the model file!")

# 3. Direct API logic verification simulation
def simulate_fastapi_endpoint_logic(data: TelemetryInput):
    """Simulates the internal request handling and matrix math of the /predict route."""
    # Convert incoming schema to matching model feature layout
    input_dataframe = pd.DataFrame([{
        "temperature_c": data.temperature_c,
        "vibration_mms": data.vibration_mms,
        "voltage_v": data.voltage_v
    }])

    # Execute high-speed matrix prediction
    prediction = int(loaded_model.predict(input_dataframe)[0])
    probability = loaded_model.predict_proba(input_dataframe)[0][prediction]

    status_msg = "CRITICAL: Severe Anomaly Detected. Trigger Emergency Shutdown Protocol!" if prediction == 1 else "OPERATIONAL: System Parameters Within Normal Variance Limits."

    return {
        "anomaly_flag": prediction,
        "confidence_score": round(float(probability) * 100, 2),
        "system_directive": status_msg
    }

# --- RUN EXECUTION TEST ---
if __name__ == "__main__":
    # Define a high-risk operational vector
    mock_payload = TelemetryInput(temperature_c=115.8, vibration_mms=18.2, voltage_v=242.0)

    print("\n📡 Emulating FastAPI Route: [POST] /predict")
    print(f"Incoming Verified JSON Payload: {mock_payload.dict()}")

    # Run the internal gateway processing pipeline
    response_payload = simulate_fastapi_endpoint_logic(mock_payload)

    print("\n⚡ [FastAPI Response Schema Generated Successfully]")
    print(f"HTTP Return Code Mock: 200 OK")
    print(json.dumps(response_payload, indent=4))

✅ Production Success: Model artifact 'predictive_model.pkl' successfully memory-mapped.

📡 Emulating FastAPI Route: [POST] /predict
Incoming Verified JSON Payload: {'temperature_c': 115.8, 'vibration_mms': 18.2, 'voltage_v': 242.0}

⚡ [FastAPI Response Schema Generated Successfully]
HTTP Return Code Mock: 200 OK
{
    "anomaly_flag": 1,
    "confidence_score": 90.0,
    "system_directive": "CRITICAL: Severe Anomaly Detected. Trigger Emergency Shutdown Protocol!"
}


/tmp/ipykernel_4511/305269735.py:49: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"Incoming Verified JSON Payload: {mock_payload.dict()}")
